# 01 — RAG Fundamentals

An end-to-end walkthrough of building a Retrieval-Augmented Generation (RAG) pipeline.

## What you'll learn
- Load and prepare documents for RAG
- Split text into chunks
- Generate embeddings
- Store and retrieve from Pinecone
- Generate answers with Gemini using retrieved context

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install langchain langchain-openai langchain-pinecone langchain-community pinecone-client python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
assert os.getenv("PINECONE_API_KEY"), "Set PINECONE_API_KEY in .env"

print("API keys loaded.")

## 1. Document Loading

LangChain provides loaders for many formats. Here we use `TextLoader` for a simple text file, but the same pattern works with PDFLoader, WebBaseLoader, etc.

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Option A: Load a single file
# loader = TextLoader("path/to/document.txt")

# Option B: Load all .txt files from a directory
# loader = DirectoryLoader("./docs", glob="**/*.txt", loader_cls=TextLoader)

# For this demo, we create sample documents inline
sample_docs = [
    """Retrieval-Augmented Generation (RAG) is a technique that combines large language models
with external knowledge retrieval. Instead of relying solely on the model's training data,
RAG fetches relevant documents at query time and injects them into the prompt context.
This approach reduces hallucination and keeps responses grounded in factual sources.""",

    """Vector databases store embeddings — numerical representations of text that capture
semantic meaning. When a user submits a query, it is converted to an embedding and compared
against stored vectors using cosine similarity. The most similar vectors are returned as
relevant results. Popular vector databases include Pinecone, Chroma, Weaviate, and FAISS.""",

    """Chunking is the process of splitting documents into smaller pieces for embedding.
Common strategies include fixed-size chunking, recursive character splitting, and semantic
chunking. The goal is to create chunks that are small enough for precise retrieval but
large enough to maintain context. Overlap between chunks helps preserve continuity.""",

    """LangChain is a framework for building applications powered by language models.
It provides abstractions for document loaders, text splitters, vector stores, and chains.
LangChain integrates with many LLM providers including Google Gemini, OpenAI, and Anthropic.
It also supports retrieval chains, agents, and tool use.""",

    """Embedding models convert text into dense vectors. Models like text-embedding-3-small
and text-embedding-3-small produce vectors where semantically similar texts are close
together in the vector space. The quality of embeddings directly impacts retrieval accuracy.
Choosing the right embedding model depends on your domain, language, and latency requirements."""
]

print(f"Loaded {len(sample_docs)} documents.")

## 2. Text Chunking

We split documents into smaller chunks. `RecursiveCharacterTextSplitter` is a solid default — it tries to split on natural boundaries (paragraphs, sentences, words) in order.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # max characters per chunk
    chunk_overlap=50,     # overlap between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Create Document objects with metadata
from langchain.schema import Document

documents = []
for i, text in enumerate(sample_docs):
    doc = Document(page_content=text, metadata={"source": f"doc_{i}", "topic": "rag-fundamentals"})
    documents.append(doc)

chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} (source: {chunk.metadata['source']}) ---")
    print(chunk.page_content[:150] + "...")

## 3. Generate Embeddings

We use Google's `text-embedding-3-small` model. Each chunk is converted to a 768-dimensional vector.

In [ ]:
from langchain_openai import ChatOpenAIEmbeddings

embedding_model = ChatOpenAIEmbeddings(
    model="models/text-embedding-3-small"
)

# Test: embed a single query
test_embedding = embedding_model.embed_query("What is RAG?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

## 4. Store in Pinecone

Create a Pinecone index and upsert our chunks.

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

INDEX_NAME = "rag-fundamentals"

# Create index if it doesn't exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,    # must match embedding dimension
        metric="cosine",
        spec={"serverless": {"cloud": "aws", "region": "us-east-1"}}
    )
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists.")

index = pc.Index(INDEX_NAME)

In [ ]:
from langchain_pinecone import PineconeVectorStore

# Upsert chunks into Pinecone
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    index_name=INDEX_NAME,
    namespace="demo"
)

print(f"Upserted {len(chunks)} chunks into Pinecone index '{INDEX_NAME}'.")

## 5. Retrieval

Given a user query, retrieve the most relevant chunks from the vector store.

In [ ]:
query = "How does chunking work in RAG?"

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3, "namespace": "demo"}
)

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} chunks for query: '{query}'\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} (score from metadata) ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(doc.page_content[:200])
    print()

## 6. Generation with Context

Combine retrieval and generation. We construct a prompt that includes the retrieved chunks as context, then pass it to Gemini.

In [ ]:
from langchain_openai import ChatChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Initialize Gemini
llm = ChatChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# RAG prompt template
rag_prompt = PromptTemplate(
    template="""Answer the question based on the following context.
If the context does not contain enough information, say "I don't have enough information to answer this."
Do not make up information.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Helper: format retrieved docs into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Run the chain
answer = rag_chain.invoke("How does chunking work in RAG?")
print("Answer:")
print(answer)

## 7. Adding Source Citations

For transparency, return the source documents alongside the answer.

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt}
)

result = qa_chain.invoke({"query": "What are vector databases used for?"})

print("Answer:")
print(result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  - {doc.metadata.get('source', 'unknown')}: {doc.page_content[:100]}...")

## 8. Interactive Querying

Wrap everything in a simple function for repeated queries.

In [ ]:
def ask_rag(question: str, k: int = 3):
    """Query the RAG pipeline and return answer with sources."""
    result = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {result['result']}")
    print("\nSources:")
    for i, doc in enumerate(result["source_documents"]):
        print(f"  [{i+1}] {doc.metadata.get('source', '?')} — {doc.page_content[:120]}...")
    print("\n" + "="*60 + "\n")

ask_rag("What is the purpose of RAG?")
ask_rag("How do embeddings work?")
ask_rag("What is the capital of France?")  # should say not enough info

## Summary

You built a complete RAG pipeline:

1. **Loaded** documents from raw text
2. **Chunked** them using recursive splitting
3. **Embedded** chunks with text-embedding-3-small
4. **Stored** vectors in Pinecone
5. **Retrieved** relevant chunks for a query
6. **Generated** grounded answers with Gemini

Next: explore different chunking strategies in `02-chunking-strategies.ipynb`.